# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library. This dataset details the analysis of protein abundance, modifications, and sequences in human (Homo sapiens) samples. It includes various fields such as accession, description, coverage percentage, peptide counts, and molecular weight (MW) alongside calculated parameters such as pI and normalized abundances across different samples.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, their names, and unique `@id`s.

In [ ]:
# List all record sets and their fields (@id is always present)
print("\nAvailable Record Sets and their Fields:")
record_sets = list(dataset.record_sets)

for rs in record_sets:
    print(f"\nRecordSet: {rs.name}\n  @id: {rs.id}")
    for field in rs.fields:
        print(f"   - Field: {field.name} (@id: {field.id}, dataType: {field.data_type})")

## 3. Data Extraction
Let's load data from a specific record set into a DataFrame for analysis. We'll use the record set and field `@id`s listed above.

In [ ]:
# Collect all record set @ids (for demonstration, extract them as variables)
record_set_ids = [rs.id for rs in record_sets]
print(f"Record Set @ids: {record_set_ids}")

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for: {record_set_id} with shape {dataframes[record_set_id].shape}")

# Select one record set for further exploration
if len(record_set_ids) > 0:
    focus_record_set_id = record_set_ids[0]
    if focus_record_set_id in dataframes:
        print(f"\nColumns in record set {focus_record_set_id}: {list(dataframes[focus_record_set_id].columns)}")
        display(dataframes[focus_record_set_id].head())
    else:
        print(f"No records loaded for record set {focus_record_set_id}")
else:
    print("No record sets found.")

## 4. Exploratory Data Analysis (EDA)
Now, apply common data processing steps such as filtering, normalization, and grouping. For demonstration, we'll select a numeric field and a categorical/group field using their `@id`s (adjust these based on the record set overview).

In [ ]:
# --- EDA Section ---
# Replace these with actual @id strings from section 2 as needed

numeric_field = None
group_field = None

if focus_record_set_id in dataframes:
    df = dataframes[focus_record_set_id]
    # Search for likely numeric and group fields (@id) in the DataFrame
    for col in df.columns:
        # Try to pick numeric field (float/int) and group field (object)
        if numeric_field is None and pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
        elif group_field is None and df[col].dtype == 'object':
            group_field = col
    print(f"Selected numeric_field @id: {numeric_field}")
    print(f"Selected group_field @id: {group_field}")

    # Proceed if a numeric field is found
    if numeric_field is not None:
        threshold = df[numeric_field].mean()  # For illustration, use mean as a threshold
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}: {len(filtered_df)} rows")
        display(filtered_df.head())

        # Normalize the numeric field
        norm_field = f"{numeric_field}_normalized"
        filtered_df[norm_field] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, norm_field]].head())

        # Group by group_field if available
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"\nGrouped by {group_field} (mean of {numeric_field}):")
            display(grouped_df.head())
    else:
        print("No numeric field found for EDA.")
else:
    print(f"No DataFrame for record set {focus_record_set_id}")

## 5. Visualization
Visualize the distribution of the selected numeric field and its relation to the group field when available.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

if focus_record_set_id in dataframes and numeric_field is not None:
    df = dataframes[focus_record_set_id]

    plt.figure(figsize=(7,4))
    df[numeric_field].hist(bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If group_field is categorical with reasonable cardinality, boxplot
    if group_field is not None and df[group_field].nunique() < 20:
        plt.figure(figsize=(10,5))
        df.boxplot(column=numeric_field, by=group_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.suptitle("")
        plt.xticks(rotation=45)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
We demonstrated how to load, explore, and process the FAIR^2 mass spectrometry dataset using the `mlcroissant` library. Key steps included discovering record sets and their schema via unique `@id`s, bulk loading the data, and performing example analysis and visualizations. Further domain-specific analyses can be performed by referencing fields via their stable `@id` and using pandas for manipulation and visualization.